$$\large{\color{yellow}{\text{MDS6304 Deep Learning ETEPW}}}$$

$$\large\text{Theme}: \underline{\text{demonstrating step-by-step the forward and backward propagation of a softmax classifier for a batch of samples}}$$

---

Load essential libraries

---

In [6]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
plt.style.use('dark_background')
%matplotlib inline
import sys
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
import seaborn as sns

---

Mount Google Drive folder if running Google Colab

---

In [2]:
## Mount Google drive folder if running in Colab
if('google.colab' in sys.modules):
    from google.colab import drive
    drive.mount('/content/drive', force_remount = True)
    DIR = '/content/drive/MyDrive/Colab Notebooks/MAHE/MSIS Coursework/EvenSem2026MAHE'
    DATA_DIR = DIR+'/Data/'
else:
    DATA_DIR = 'Data/'

---

**The patient data matrix with output labels and initial weight matrix**

![patient dataset](https://1drv.ms/i/s!AjTcbXuSD3I3hspfrgklysOtJMOjaA?embed=1&width=800)

---



In [3]:
# Patients data matrix
X = torch.tensor([[72, 120, 37.3, 104, 32.5],
                 [85, 130, 37.0, 110, 14],
                 [68, 110, 38.5, 125, 34],
                 [90, 140, 38.0, 130, 26],
                 [84, 132, 38.3, 146, 30],
                 [78, 128, 37.2, 102, 12]], dtype = torch.float64)
print(f'Patient data matrix X:\n {X}') #f-string in Python
print('---------')

# Initial Weights matrix (trainable tensor)
W = torch.tensor([[-0.1, 0.5, 0.3],
                  [0.9, 0.3, 0.5],
                  [-1.5, 0.4, 0.1],
                  [0.1, 0.1, -1.0],
                  [-1.2, 0.5, -0.8]], dtype = torch.float64,
                 requires_grad = True)
print(f'Initial weights matrix:\n {W}')
print('---------')

# Create a 1D-numpy array of output labels (equivalent to a rank-1 tensor in
# PyTorch which itself is equivalent to a vector in pen & paper)
y = np.array(['non-diabetic',
              'diabetic',
              'non-diabetic',
              'pre-diabetic',
              'diabetic',
              'pre-diabetic'])
# Creating a one-hot encoder object
ohe = OneHotEncoder(sparse_output = False)
# Create the one-hot encoded true output labels matrix
Y = torch.tensor(ohe.fit_transform(y.reshape(-1, 1)), dtype = torch.float64)
print(f'One-hot encoded output labels matrix:\n {Y}')
print('---------')

# Standardize the data
sc = StandardScaler() # create a standard scaler object
X_std = torch.tensor(sc.fit_transform(X), dtype = torch.float64)
print(f'The standardized data matrix:\n{X_std}')

Patient data matrix X:
 tensor([[ 72.0000, 120.0000,  37.3000, 104.0000,  32.5000],
        [ 85.0000, 130.0000,  37.0000, 110.0000,  14.0000],
        [ 68.0000, 110.0000,  38.5000, 125.0000,  34.0000],
        [ 90.0000, 140.0000,  38.0000, 130.0000,  26.0000],
        [ 84.0000, 132.0000,  38.3000, 146.0000,  30.0000],
        [ 78.0000, 128.0000,  37.2000, 102.0000,  12.0000]],
       dtype=torch.float64)
---------
Initial weights matrix:
 tensor([[-0.1000,  0.5000,  0.3000],
        [ 0.9000,  0.3000,  0.5000],
        [-1.5000,  0.4000,  0.1000],
        [ 0.1000,  0.1000, -1.0000],
        [-1.2000,  0.5000, -0.8000]], dtype=torch.float64, requires_grad=True)
---------
One-hot encoded output labels matrix:
 tensor([[0., 1., 0.],
        [1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [0., 0., 1.]], dtype=torch.float64)
---------
The standardized data matrix:
tensor([[-0.9799, -0.7019, -0.7238, -0.9871,  0.8920],
        [ 0.7186,  0.3509, 

---

Forward propagation for all the samples in the batch

---

In [ ]:
# Raw scores
Z = torch.matmul(X_std, W)

# Softmax-activated scores
A = F.softmax(Z, dim = -1)
# The following also works
#softmax = torch.nn.Softmax(dim = -1)
#A = softmax(Z)

# Calculate the average training loss
L = torch.mean(-torch.log(torch.sum(Y*A, dim = 1)))
# The following also works
#cce = torch.nn.CrossEntropyLoss()
#L = cce(Z, torch.argmax(Y, dim = -1).long())
#L = F.cross_entropy(Z, torch.argmax(Y, dim = -1).long())
#print(L)

tensor(1.2304, dtype=torch.float64, grad_fn=<NllLossBackward0>)


---

Backward propagation for all the samples in the batch

---

In [ ]:
# Loss layer
grad = -Y/A # has shape 6x3
print(grad)

# Softmax layer
I = torch.eye(3)
softmax_local_gradient = (I-A.unsqueeze(-1)) * (A.unsqueeze(-2)) # has 6x3x3
print(softmax_local_gradient)
grad = torch.einsum('ij, ijk -> ik', grad, softmax_local_gradient)
print(grad) # has shape 6x3

# Dense layer
grad = (1/6)*(torch.einsum('ij, ik -> jk', X_std, grad))
print(grad)


---

Update weights

---

In [24]:
with torch.no_grad():
    W -= 0.01*grad

W

tensor([[-0.0970,  0.4927,  0.3044],
        [ 0.9016,  0.2932,  0.5052],
        [-1.4949,  0.3948,  0.1001],
        [ 0.1083,  0.0915, -0.9997],
        [-1.1964,  0.5004, -0.8041]], dtype=torch.float64, requires_grad=True)